# IaC 第4周:综合实战 — IaC 全流程

> **学习目标**:串联 Terraform + Ansible + Helm + CI/CD,实现完整的 IaC 流水线

---

## Day 22-23:CI/CD 中运行 IaC

```yaml
# .github/workflows/terraform.yml
name: Terraform CI
on:
  pull_request:
    paths: ['terraform/**']
  push:
    branches: [main]

jobs:
  terraform:
    steps:
      - uses: actions/checkout@v4
      - uses: hashicorp/setup-terraform@v3
      - run: terraform fmt -check -recursive
      - run: terraform init && terraform validate
      - if: github.event_name == 'pull_request'
        run: terraform plan          # PR: 只看不改
      - if: github.event_name == 'push'
        run: terraform apply -auto-approve  # merge: 执行
```

模式:PR → plan(预览变更);merge → apply(执行变更)

## Day 24:Terraform + Ansible 协作模式

| 模式 | 做法 | 推荐度 |
|------|------|--------|
| local-exec | Terraform provisioner 调 ansible | 适合小项目 |
| CI Pipeline (推荐) | CI 中 step 1: TF apply → step 2: ansible-playbook | 生产级 |
| Dynamic inventory | TF output → 生成 inventory → Ansible 读取 | 大型、变化频繁 |

## Day 25:K8s 的 IaC — Terraform Helm Provider

```hcl
provider "helm" {
  kubernetes { config_path = "~/.kube/config" }
}

resource "helm_release" "prometheus" {
  name       = "monitoring"
  repository = "https://prometheus-community.github.io/helm-charts"
  chart      = "kube-prometheus-stack"
  namespace  = "monitoring"
  create_namespace = true

  values = [file("./values/prometheus-${var.environment}.yaml")]
}
```

优势:统一管理基础设施 + K8s 应用(一个 `terraform apply`),Terraform state 中能看到 Helm Release 状态

## Day 26:IaC 项目结构设计

```
iac-repo/
├── modules/                  # 共享 Module
├── environments/
│   ├── dev/main.tf + tfvars
│   ├── staging/main.tf + tfvars
│   └── prod/main.tf + tfvars
├── ansible/                  # 配置管理
│   ├── site.yml
│   ├── inventory/
│   └── roles/
├── .github/workflows/        # CI/CD
│   ├── terraform-dev.yml
│   ├── terraform-prod.yml
│   └── ansible.yml
└── README.md
```

IaC 仓库和 App 仓库应独立维护 —— 基础设施的变化节奏和 App 代码不同。

## Day 28:第4周综合练习

In [ ]:
print("""
IaC 全流程项目 - 你现在能构建的:

阶段一:基础设施 (Terraform)
  ├── Docker 网络
  └── PostgreSQL 数据库

阶段二:K8s 中间件 (Terraform + Helm)
  ├── Prometheus Stack
  ├── Loki Stack
  ├── Jaeger
  └── ingress-nginx

阶段三:应用部署 (Terraform + Helm)
  ├── Python Web 应用
  ├── Python Worker 应用
  └── ServiceMonitor (Prometheus 自动发现)

阶段四:主机配置 (Ansible)
  ├── 安全基线 (SSH, 防火墙)
  ├── Docker 安装与配置
  └── 监控 agent

阶段五:CI/CD
  ├── PR → terraform plan + ansible --check
  └── merge → terraform apply + ansible-playbook

产出物:
  ├── IaC 仓库 (完整可运行的代码)
  ├── CI/CD Workflow 文件
  ├── README (初始化/部署/销毁)
  └── 架构图 (资源拓扑 + 数据流向)

══════════════════════════════════
  IaC 4 周学习完成!
══════════════════════════════════

你现在能用代码管理一切基础设施,
从虚拟机到 K8s,从 CI 到监控,
掌握了 Terraform + Ansible + Helm + CI/CD。
""")